In [1]:
import numpy as np
from scipy.stats import norm, spearmanr

def compare_spearman_rhos(n, r1, r2):
    # n = len(x)
    # r1, _ = spearmanr(x, y1)
    # r2, _ = spearmanr(x, y2)
    z1 = 0.5 * np.log((1 + r1) / (1 - r1))
    z2 = 0.5 * np.log((1 + r2) / (1 - r2))
    se = np.sqrt(1.060 / (n - 3))
    z = (z1 - z2) / se
    p = 2 * (1 - norm.cdf(abs(z)))
    return r1, r2, z, p

In [2]:
import numpy as np
from scipy.stats import kendalltau, norm

def tau_to_r(tau: float) -> float:
    """
    Approximate conversion from Kendall's tau to Pearson's r.
    This uses a meta-analysis formula (e.g., Walker, 2003).
    """
    return np.sin(np.pi * tau / 2)

def fisher_z(r: float) -> float:
    """Fisher's z-transformation for Pearson's r."""
    return 0.5 * np.log((1 + r) / (1 - r))

def compare_kendall_taus(n, tau1, tau2):
    """
    Compare two Kendall's tau correlations via Fisher's z on
    approximate Pearson r values.
    
    Parameters:
    - n: length of the two arrays
    - tau1, tau2: Kendall's tau values
    Return:
    - z_stat: test statistic
    - p_value: two-sided p-value
    """
    # # Compute Kendall's tau correlations
    # tau1, _ = kendalltau(x, y1)
    # tau2, _ = kendalltau(x, y2)
    
    # Convert to Pearson r via approximation
    r1 = tau_to_r(tau1)
    r2 = tau_to_r(tau2)
    
    # Fisher's z-transform
    z1 = fisher_z(r1)
    z2 = fisher_z(r2)
    
    # Standard error assuming similar sample size
    se = np.sqrt(1 / (n - 3))
    
    # Difference statistic
    z_stat = (z1 - z2) / se
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    return {
        "z_stat": z_stat, "p_value": p_value
    }

In [20]:
compare_spearman_rhos(97, 0.4434, 0.3789)[-1]<=0.05

False

In [21]:
compare_kendall_taus(97, 0.3097, 0.2612)['p_value']<=0.05

False

In [ ]:
import numpy as np
from scipy.stats import kendalltau

def compare_kendall_tau(x, y1, y2, num_bootstrap=10000, alpha=0.05):
    diffs = []
    n = len(x)
    for _ in range(num_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        t1, _ = kendalltau(x[idx], y1[idx])
        t2, _ = kendalltau(x[idx], y2[idx])
        diffs.append(t1 - t2)
    diffs = np.array(diffs)
    lower = np.percentile(diffs, 100 * (alpha / 2))
    upper = np.percentile(diffs, 100 * (1 - alpha / 2))
    p_value = np.mean(np.abs(diffs) >= np.abs(np.mean(diffs)))
    return np.mean(diffs), (lower, upper), p_value